In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=SK1byu8XEpDCBYAhGObOV3OYOcGZws&access_type=offline&code_challenge=M2zQHUIYG0gPvyeK2O1XDw-FrrNLHKgdbwTL_m8zckg&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


Updates are available for some Google Clo

In [2]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize": "2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()


session = GentropySession()


Loading BokehJS ...

/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/03 16:55:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
path_to_release_folder = "gs://open-targets-data-releases/25.06/"


si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study/")
sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set/")

sl_eff = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/ss60/gentropy-manuscript/chapters/variant-effect-prediction/25.07/lead_variant_effect"
)

l2g_full = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_prioritised_genes_per_CS.parquet"
)

In [4]:
disease_index_path = path_to_release_folder + "output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

platform_chembl_evidence_path = (
    path_to_release_folder + "output/evidence/sourceId=chembl"
)
chembl_evidence = session.spark.read.parquet(platform_chembl_evidence_path)

In [5]:
efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig,
    efo_ids="MONDO_0045024",
)

In [6]:
l2g_full = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/l2g_full_for_enrichment"
)
l2g_full.count()

70400

In [7]:
pharmprj = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/pharmacoproject_processed.parquet"
)
pharmprj.count()

6047

In [8]:
pharmprj.show(1)

+-----------+-------------+---------------+
|  diseaseId|clinicalPhase|       targetId|
+-----------+-------------+---------------+
|EFO_0000474|          0.5|ENSG00000183044|
+-----------+-------------+---------------+
only showing top 1 row



In [9]:
chembl_evidence.groupBy("clinicalPhase").count().show()

+-------------+------+
|clinicalPhase| count|
+-------------+------+
|          1.0|126478|
|          4.0|104823|
|          0.5|  7183|
|          3.0|122304|
|          2.0|212315|
+-------------+------+



In [10]:
chembl_evidence = chembl_evidence.select("diseaseId", "clinicalPhase", "targetId")

In [11]:
chembl_evidence.select("diseaseId", "targetId").distinct().count()

74187

In [12]:
chembl_evidence.printSchema()

root
 |-- diseaseId: string (nullable = true)
 |-- clinicalPhase: double (nullable = true)
 |-- targetId: string (nullable = true)



In [14]:
rusina_df = session.spark.read.csv(
    "/Users/yt4/Desktop/paper_old/drug target enrichemnt/RusinaP_etal_supplementary_table.txt",
    sep="\t",
    header=True,
    nullValue="NA",
)
rusina_df = rusina_df.select("targetIds", "diseaseIds", "relatedIds").withColumn(
    "clinicalPhase", f.lit(4.0)
)
rusina_df.count()

428

In [15]:
rusina_df.printSchema()

root
 |-- targetIds: string (nullable = true)
 |-- diseaseIds: string (nullable = true)
 |-- relatedIds: string (nullable = true)
 |-- clinicalPhase: double (nullable = false)



In [16]:
rusina_df.show(2)

+--------------------+-----------+-----------+-------------+
|           targetIds| diseaseIds| relatedIds|clinicalPhase|
+--------------------+-----------+-----------+-------------+
|ENSG00000123201, ...|EFO_0001361|EFO_0000537|          4.0|
|ENSG00000133019, ...|EFO_0000341|       NULL|          4.0|
+--------------------+-----------+-----------+-------------+
only showing top 2 rows



In [17]:
rusina_df = rusina_df.withColumn(
    "targetIds", f.split(f.trim(f.col("targetIds")), r"\s*,\s*")
)
rusina_df.select("targetIds").show(2, truncate=False)


+--------------------------------------------------------------------+
|targetIds                                                           |
+--------------------------------------------------------------------+
|[ENSG00000123201, ENSG00000164116, ENSG00000061918, ENSG00000152402]|
|[ENSG00000133019, ENSG00000169252]                                  |
+--------------------------------------------------------------------+
only showing top 2 rows



In [18]:
rusina_df = rusina_df.withColumn(
    "diseaseIds", f.split(f.trim(f.col("diseaseIds")), r"\s*,\s*")
)
rusina_df.select("diseaseIds").show(2, truncate=False)


+-------------+
|diseaseIds   |
+-------------+
|[EFO_0001361]|
|[EFO_0000341]|
+-------------+
only showing top 2 rows



In [19]:
rusina_df = rusina_df.withColumn(
    "relatedIds", f.split(f.trim(f.col("relatedIds")), r"\s*,\s*")
)
rusina_df.select("relatedIds").show(2, truncate=False)


+-------------+
|relatedIds   |
+-------------+
|[EFO_0000537]|
|NULL         |
+-------------+
only showing top 2 rows



In [20]:
rusina_df.show(10)

+--------------------+--------------------+-------------+-------------+
|           targetIds|          diseaseIds|   relatedIds|clinicalPhase|
+--------------------+--------------------+-------------+-------------+
|[ENSG00000123201,...|       [EFO_0001361]|[EFO_0000537]|          4.0|
|[ENSG00000133019,...|       [EFO_0000341]|         NULL|          4.0|
|[ENSG00000168356,...|        [HP_0001250]|         NULL|          4.0|
|[ENSG00000113580,...|       [EFO_0000341]|         NULL|          4.0|
|[ENSG00000178394,...|     [MONDO_0002009]|         NULL|          4.0|
|                NULL|                NULL|         NULL|          4.0|
|[ENSG00000091831,...|[EFO_0003922, EFO...|         NULL|          4.0|
|   [ENSG00000156738]|       [EFO_0000095]|         NULL|          4.0|
|[ENSG00000141736,...|       [EFO_0003060]|         NULL|          4.0|
|   [ENSG00000010671]|       [EFO_1001469]|         NULL|          4.0|
+--------------------+--------------------+-------------+-------

In [21]:
rusina_df = rusina_df.withColumn(
    "diseaseIds",
    f.when(f.col("relatedIds").isNull(), f.col("diseaseIds")).otherwise(
        f.concat(f.col("diseaseIds"), f.col("relatedIds"))
    ),
).drop("relatedIds")

rusina_df.select("diseaseIds").show(2, truncate=False)

+--------------------------+
|diseaseIds                |
+--------------------------+
|[EFO_0001361, EFO_0000537]|
|[EFO_0000341]             |
+--------------------------+
only showing top 2 rows



In [22]:
rusina_df.show(10)

+--------------------+--------------------+-------------+
|           targetIds|          diseaseIds|clinicalPhase|
+--------------------+--------------------+-------------+
|[ENSG00000123201,...|[EFO_0001361, EFO...|          4.0|
|[ENSG00000133019,...|       [EFO_0000341]|          4.0|
|[ENSG00000168356,...|        [HP_0001250]|          4.0|
|[ENSG00000113580,...|       [EFO_0000341]|          4.0|
|[ENSG00000178394,...|     [MONDO_0002009]|          4.0|
|                NULL|                NULL|          4.0|
|[ENSG00000091831,...|[EFO_0003922, EFO...|          4.0|
|   [ENSG00000156738]|       [EFO_0000095]|          4.0|
|[ENSG00000141736,...|       [EFO_0003060]|          4.0|
|   [ENSG00000010671]|       [EFO_1001469]|          4.0|
+--------------------+--------------------+-------------+
only showing top 10 rows



In [23]:
rusina_df.count()

428

In [24]:
rusina_df = rusina_df.na.drop()

In [25]:
rusina_df.count()

352

In [26]:
rusina_df = rusina_df.withColumn("targetId", f.explode("targetIds")).drop("targetIds")
rusina_df = rusina_df.withColumn("diseaseId", f.explode("diseaseIds")).drop(
    "diseaseIds"
)

In [27]:
rusina_df = rusina_df.distinct()

In [28]:
rusina_df.count()

1036

In [29]:
chembl_evidence.count()

573103

In [30]:
chembl_evidence.distinct().count()

110502

In [31]:
chembl_evidence_new = (
    chembl_evidence.unionByName(rusina_df).unionByName(pharmprj).distinct().cache()
)
chembl_evidence_new.count()

113676

In [33]:
chembl_evidence_new.count()

113676

In [34]:
chembl_evidence_new.select("diseaseId", "targetId").distinct().count()

74682

In [ ]:
efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig,
    efo_ids=["MONDO_0045024"],
)

In [36]:
chembl_processed_new = chemblDrugEnrichment.process_chembl_evidence(
    chembl_evidence_new, efo_to_remove=efo_to_remove
)

In [38]:
chembl_processed_new.show(1)

+---------------+-----------+----------------+
|       targetId|  diseaseId|maxClinicalPhase|
+---------------+-----------+----------------+
|ENSG00000137252|EFO_0004698|             4.0|
+---------------+-----------+----------------+
only showing top 1 row



In [39]:
chembl_processed_new.count()

37671

In [40]:
chembl_processed_new.groupBy("maxClinicalPhase").count().show()

+----------------+-----+
|maxClinicalPhase|count|
+----------------+-----+
|             1.0| 6090|
|             4.0| 4977|
|             3.0|12216|
|             2.0|14388|
+----------------+-----+



In [ ]:
def process_chembl_evidence_mod(
    chembl_orig: DataFrame, efo_to_remove: list[str] | None = None
) -> DataFrame:
    """Process chembl evidence.

    Removes EFO from the list, usualy oncolgy.

    Args:
        chembl_orig (DataFrame): Chembl evidence
        efo_to_remove (list[str] | None): List of EFO IDs to remove
    Returns:
        DataFrame: Processed chembl evidence
    """
    if efo_to_remove is not None:
        chembl_orig = chembl_orig.filter(~f.col("diseaseId").isin(efo_to_remove))

    chembl_evidence_max = chembl_orig.groupBy("targetId", "diseaseId").agg(
        f.max("clinicalPhase").alias("maxClinicalPhase")
    )

    return chembl_evidence_max

In [42]:
chembl_processed_new2 = process_chembl_evidence_mod(
    chembl_evidence_new, efo_to_remove=efo_to_remove
)

In [43]:
chembl_processed_new2.groupBy("maxClinicalPhase").count().show()

+----------------+-----+
|maxClinicalPhase|count|
+----------------+-----+
|             1.0| 6090|
|             4.0| 4977|
|             0.5| 1055|
|             3.0|12216|
|             2.0|14388|
+----------------+-----+



In [45]:
chembl_processed_new.select("diseaseId", "targetId").distinct().count()

37671

In [46]:
pharmprj.select("diseaseId", "targetId").distinct().count()

6047

In [47]:
pharmprj.groupBy("clinicalPhase").count().show()

+-------------+-----+
|clinicalPhase|count|
+-------------+-----+
|          1.0|  885|
|          4.0| 1018|
|          0.5| 1168|
|          3.0|  697|
|          2.0| 2279|
+-------------+-----+



In [115]:
evidence = chemblDrugEnrichment.to_disease_target_evidence(
    table_with_score=l2g_full.drop("diseaseIds"),
    score_column="score",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.1,
)

In [116]:
enrich = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.1,
    efo_ancestors_to_remove=["MONDO_0045024"],
)
enrich

,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,Relative success,ci_rs_low,ci_rs_high,rs_p_value,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.746148,1.154068e-06,1.375083,2.217346,1.076425,1.050129,1.103380,5.340474e-09,6087,30548,76,666,151704
1,3+,2.060942,7.941172e-22,1.773924,2.394399,1.398926,1.321405,1.480995,8.135878e-31,20294,16341,279,463,151704
2,4+,3.618578,3.534571e-49,3.093638,4.232591,2.764540,2.483639,3.077211,3.161170e-77,32313,4322,500,242,151704


In [117]:
enrich = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence_new,
    indirect_assoc_score_thr=0.1,
    efo_ancestors_to_remove=["MONDO_0045024"],
)
enrich

25/12/04 12:25:30 WARN CacheManager: Asked to cache already cached data.        


,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,Relative success,ci_rs_low,ci_rs_high,rs_p_value,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.825495,2.687302e-07,1.429452,2.331265,1.079566,1.054123,1.105623,3.140451e-10,6018,30906,72,675,151704
1,3+,2.148395,2.873782e-24,1.847679,2.498052,1.413545,1.337961,1.493399,5.236916e-35,20209,16715,269,478,151704
2,4+,3.712885,5.219076e-54,3.185417,4.327696,2.757746,2.493139,3.050438,1.743412e-86,32210,4714,484,263,151704


In [ ]:
32210 + 4714 + 484 + 263

37671

In [ ]:
32313 + 4322 + 500 + 242

37377

In [ ]:
chembl_evidence_new.write.parquet(
    "gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/chembl_evidence_with_rusina_et_al_pharmprj.parquet"
)